In [ ]:
!pip install bert-score
!pip install datasets
!pip install sentence-transformers datasets accelerate transformers wandb
!pip install sentence-transformers torch sklearn bert-score
!pip install sentencepiece
!pip install -q sentencepiece transformers sentence-transformers bert-score torch scikit-learn pandas numpy
!pip install -q pytorch-crf

  Using cached sklearn-0.0.post12.tar.gz (2.6 kB)
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [ ]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["WANDB_DISABLED"] = "true"  # sentence-transformers → W&B өшіру

# !pip install -q -U "transformers>=4.41.0" "tokenizers>=0.15.2" \
#                   "sentence-transformers>=2.7.0" "sentencepiece>=0.1.99" "bert-score"

import time, random
from pathlib import Path
import json  # ✅ JSON датасетін оқу үшін

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

# bert_score және т.б. енді қолданылмайды, бірақ құрылымды бұзбау үшін қалдыруға да болады
# from bert_score import score as bertscore

from transformers import (
    set_seed,
)

from sentence_transformers import SentenceTransformer, InputExample, losses, models
from google.colab import drive
from sklearn.model_selection import train_test_split

# --------------------------
# 🔒 Seed (strict deterministic емес, бірақ тұрақты)
# --------------------------
SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

try:
    torch.use_deterministic_algorithms(False)
except Exception:
    pass

torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("🖥 DEVICE:", DEVICE)

# =========================
# 0) Google Drive-ты монттау
# =========================
drive.mount("/content/drive", force_remount=True)

# =========================
# 1) Жолдар
# =========================
PROJECT_DIR     = "/content/kz_minilm_st_direct_spm"
LOCAL_ST_DIR    = f"{PROJECT_DIR}/st_finetune_work"   # жұмыс каталог (checkpoint)
ST_OUT_DIR      = f"{PROJECT_DIR}/st_finetuned"       # финетюннен кейінгі финал модель

Path(PROJECT_DIR).mkdir(parents=True, exist_ok=True)
Path(LOCAL_ST_DIR).mkdir(parents=True, exist_ok=True)
Path(ST_OUT_DIR).mkdir(parents=True, exist_ok=True)

# =========================
# 2) QA деректері (JSON файлдан оқу)
# =========================
# ⚠ МЫНАНЫ ӨЗ ЖОЛЫҢЫЗҒА АУЫСТЫРЫҢЫЗ:
DATA_JSON_PATH = "qa_data.json"

if not Path(DATA_JSON_PATH).is_file():
    raise FileNotFoundError(f"QA JSON файлы табылмады: {DATA_JSON_PATH}")

print(f"📥 QA JSON файлдан жүктеу: {DATA_JSON_PATH}")

with open(DATA_JSON_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

# Күтілетін формат: [{"question": "...", "answer": "..."}, ...]
qa_data = pd.DataFrame(data)

required_cols = {"question", "answer"}
if not required_cols.issubset(qa_data.columns):
    raise ValueError(
        f"JSON деректерінде міндетті бағандар жоқ: {required_cols}. "
        f"Бар бағандар: {set(qa_data.columns)}"
    )

print(f"📚 Барлық QA жазба саны: {len(qa_data)}")

# --------------------------
# TRAIN / TEST SPLIT (held-out)
# --------------------------
train_df, test_df = train_test_split(
    qa_data,
    test_size=0.1,      # 5944 → ~595 тест болғанда 0.1, мұнда тек құрылым сақталады
    random_state=SEED,
    shuffle=True,
)

train_questions = train_df["question"].tolist()
train_answers   = train_df["answer"].tolist()
test_questions  = test_df["question"].tolist()
test_answers    = test_df["answer"].tolist()

print(f"📊 TRAIN: {len(train_df)} | TEST: {len(test_df)}")

# ============================================================
# 3) MiniLM стандартты токенизаторын пайдалану
#    (kazakh_bpe.model ЕНДІ ҚОЛДАНЫЛМАЙДЫ)
# ============================================================
print("🔤 MiniLM-нің стандартты токенизаторын қолданамыз (kazakh_bpe.model орнына).")

# ============================================================
# 4) MiniLM encoder + Pooling моделін құрамыз
#    (модель өз стандартты токенизаторын ТІКЕЛЕЙ қолданады)
# ============================================================
SENTENCE_T_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
print("⬇️ MiniLM encoder-ін жүктеу (стандартты токенизаторымен):", SENTENCE_T_NAME)

# Transformer модулі: HF моделін және өз токенизаторын жүктейміз
word_embedding_model = models.Transformer(
    SENTENCE_T_NAME
)

# Mean Pooling – paraphrase-multilingual-MiniLM-L12-v2 үшін стандарт
pooling_model = models.Pooling(
    word_embedding_model.get_word_embedding_dimension(),
    pooling_mode_mean_tokens=True,
    pooling_mode_cls_token=False,
    pooling_mode_max_tokens=False,
)

# Толық SentenceTransformer: (стандартты токенизатор → TRANSFORMER → POOLING)
st_model = SentenceTransformer(
    modules=[word_embedding_model, pooling_model],
    device=DEVICE
)

print("✅ Стандартты токенизатормен біріктірілген ST моделі дайын.")

# Бастапқы конфигурацияны LOCAL_ST_DIR-ге checkpoint ретінде сақтаймыз
st_model.save(LOCAL_ST_DIR)
print(f"💾 Бастапқы (pre-finetune) модель сақталды: {LOCAL_ST_DIR}")

# ============================================================

# ============================================================

# Hyperparameters (тек лог үшін, құрылым сақтау үшін)
BATCH_SIZE = 16
EPOCHS     = 5
WARMUP_PCT = 0.1
LR         = 1e-5

# MNRL үшін InputExample(texts=[q, a]) – label қажет емес
train_examples = [
    InputExample(texts=[row["question"], row["answer"]])
    for _, row in train_df.iterrows()
]

print("\n🧪 Fine-tune конфигурациясы (MNRL, q→a, бірақ бұл нұсқада іске қосылмайды):")
print(f"- TRAIN samples: {len(train_examples)}")
print(f"- BATCH_SIZE: {BATCH_SIZE}")
print(f"- EPOCHS: {EPOCHS}")
print(f"- LR: {LR}")

DO_FINETUNE = False  # <<< fine-tune-ды өшіру флагы

if DO_FINETUNE:
    train_dataloader = DataLoader(
        train_examples,
        shuffle=True,
        batch_size=BATCH_SIZE,
        drop_last=False
    )

    train_loss = losses.MultipleNegativesRankingLoss(st_model)

    num_train_steps = len(train_dataloader) * EPOCHS
    warmup_steps = int(num_train_steps * WARMUP_PCT)

    print(f"- TOTAL STEPS: {num_train_steps}")
    print(f"- WARMUP STEPS: {warmup_steps}")

    print("🚀 Fine-tune басталды (MultipleNegativesRankingLoss, MLM жоқ)...")
    t_ft0 = time.time()
    st_model.fit(
        train_objectives=[(train_dataloader, train_loss)],
        epochs=EPOCHS,
        warmup_steps=warmup_steps,
        show_progress_bar=True,
        optimizer_params={"lr": LR},
    )
    print(f"✅ Fine-tune аяқталды (t={time.time()-t_ft0:.2f}s)")
else:
    print("⚠ Fine-tune өшірілген. Pretrained MiniLM (стандартты токенизатормен) ғана қолданылады.")

# Fine-tune жасалмаса да, ағымдағы st_model-ді ST_OUT_DIR-ге сақтаймыз
st_model.save(ST_OUT_DIR)
print(f"💾 ST модель (pretrained, fine-tune ЖОҚ) сақталды: {ST_OUT_DIR}")

# ============================================================
# 6) QA іздеу (held-out TEST, эксперттік бағалау үшін дайындық)
# ============================================================

# 🔹 6.1. Retrieval моделі (pretrained/fine-tuned, стандартты токенизатормен)
retr_model = SentenceTransformer(ST_OUT_DIR, device=DEVICE)
retr_model.to(DEVICE)
retr_model.eval()

def _encode_retr(texts, batch_size=32, normalize=False):
    """Retrieval үшін эмбеддинг (pretrained немесе fine-tuned модель)."""
    with torch.inference_mode():
        vecs = retr_model.encode(
            texts,
            batch_size=batch_size,
            convert_to_numpy=True,
            normalize_embeddings=normalize
        )
    return vecs

# 🔹 6.2. Интерактив режим үшін (қаласаңыз толық база)
qa_questions = qa_data["question"].tolist()
qa_answers   = qa_data["answer"].tolist()
qa_q_emb     = _encode_retr(qa_questions, normalize=True)

# 🔹 6.3. Held-out TRAIN KB эмбеддингтері
train_q_emb = _encode_retr(train_questions, normalize=True)

def ask_question(question, threshold=0.6):
    """
    Интерактив режим: толық QA базасынан іздеу.
    Эксперттік бағалау үшін алынған жауапты қолмен тексеруге болады.
    """
    qv = _encode_retr([question], normalize=True)
    sims = (qa_q_emb @ qv.T).squeeze(1)
    idx = int(np.argmax(sims))
    if float(sims[idx]) < threshold:
        return "Кешіріңіз, нақты жауап табылмады.", -1
    return qa_answers[idx], idx

def ask_question_eval(question):
    """
    Held-out бағалау: TEST сұрағы → TRAIN KB (train_questions/train_answers).
    Ешқандай автоматты метрика есептелмейді – эксперттер pred vs true-ды өздері салыстырады.
    """
    qv = _encode_retr([question], normalize=True)
    sims = (train_q_emb @ qv.T).squeeze(1)
    idx = int(np.argmax(sims))
    return train_answers[idx], idx

# ============================================================

# ============================================================
if __name__ == "__main__":
    try:
        while True:
            user_input = input("\nСұрақ енгізіңіз (шығу үшін 'exit'): ")
            if user_input.strip().lower() == "exit":
                print("Бағдарлама тоқтатылды. 👋")
                break
            answer, idx = ask_question(user_input)
            print("\n=== Жауап ===")
            print(answer)
            print(f"(idx={idx})")
    except EOFError:
        pass



🖥 DEVICE: cuda


ValueError: mount failed